## 第 3 课：混合精度 GEMM 与自定义 FP8 MMA

> 对应原文：笔记 (2)。在 Minimal GEMM 基础上支持多种精度组合，并用 PTX 文档实现 CUTLASS 尚未封装的 FP8 MMA op。

## 学习目标

- 理清 MMA 场景下的精度体系：ComputeType / AccType / OutType；
- 掌握两种精度转换方式：特定 MMA 指令、寄存器内转换；
- 理解 **TV Layout** 与 **MN Layout**（CuTe 四大 Layout 之二）；
- 会写自定义 MMA op（`MMA op` + `MMA_Traits`）。

## 1. MMA 场景下的算子精度

一个 MMA 算子涉及三种精度：

- **输入精度**：ComputeTypeA / B / C；
- **累加器精度**：AccType（`AB` 的结果与 `C` 相加后的精度，注意它不是 Tensor Core 实际的累加精度）；
- **输出精度**：OutType（`D` 的精度，如果与 AccType 不同需要额外转换）。

例如 `mma.sync.aligned.m16n8k16.row.col.f32.bf16.bf16.f32`：AccType=FP32，ComputeTypeA/B=BF16，ComputeTypeC=FP32。

![MMA 场景下的数值精度](assets/figs/fig_01_MMA_场景下的数值精度.png)

## 2. 为什么需要混合精度算子？

两个例子（都绕不开混合精度）：

1. **PyTorch 控制精度的手段有限**：`torch.matmul` 算 BF16×BF16 累加是 FP32，但 FP16×FP16 累加却是 FP16；`torch.addmm` 要求 A、B、C 精度相同，无法直接做 BF16×BF16+FP32。
2. **低精度训练必然用混合精度**：DeepSeek V3 的 FP8 Linear 中，权重是 FP8、输入是 BF16，量化后的输入×权重累加 FP32，最后转回 BF16 输出——即 `BF16 = FP8 * FP8 + FP32`。

![DeepSeek V3 中的 FP8 精度计算](assets/figs/fig_02_DeepSeek_V3_中的_FP8_精度计算.png)

## 3. 实现混合精度：两种方式

### 3.1 方式一：使用特定 MMA 指令

`FP32 = BF16 * BF16 + FP32` 有现成指令，换 MMA op 即可：

In [ ]:
using MMA_op = SM80_16x8x8_F32BF16BF16F32_TN;

### 3.2 方式二：在寄存器中转换精度

`BF16 = BF16 * BF16 + FP32` 没有现成指令，先算出 FP32 结果再转：

In [ ]:
auto tCrO = make_tensor_like<OutType>(tCrC);  // 同 shape、精度为 OutType 的寄存器 Tensor
copy(tCrC, tCrO);                              // 等价于逐元素赋值，完成精度转换

In [ ]:
// 等价循环
for (int i = 0; i < size(tCrC); ++i) {
    tCrO(i) = tCrC(i);
}

kernel 侧增加输出矩阵，按 `cvt_out_precision` 选择路径：

In [ ]:
if constexpr (!cvt_out_precision) {
    copy(copy_atom, tCrC, tCgC);
} else {
    auto tCrO = make_tensor_like<OutType>(tCrC);
    copy(tCrC, tCrO);
    Tensor tCgO = thr_mma.partition_C(gO);
    copy(copy_atom, tCrO, tCgO);
}

### 3.3 PTX / SASS 中的精度转换

替换 MMA op 后 PTX 变成 `...f32.bf16.bf16.f32`，SASS 变成 `HMMA.1688.F32.BF16`。

寄存器转换后，PTX 多 4 条 `cvt` 指令（每线程 4 个 D 元素各转一次，`.rn` = rounds to nearest even，可换其他舍入方式）：

In [ ]:
cvt.rn.bf16.f32 %rs2, %f2;
cvt.rn.bf16.f32 %rs1, %f1;
...

SASS 侧只多 2 条打包指令（4 个 FP32 打包成 2 个寄存器，每寄存器 2 个 BF16）：

In [ ]:
F2FP.BF16.F32.PACK_AB R5, R5, R4   // (R4, R5) -> (R5)
F2FP.BF16.F32.PACK_AB R7, R7, R6

![PTX 浮点数 rounding 方式](assets/figs/fig_03_PTX_浮点数_rounding_方式.png)

## 4. 自定义 FP8 GEMM 算子

场景：Ada 起有 FP8 MMA 指令，最小 shape 为 (16, 8, 32)，PTX 支持混合 E4M3 / E5M2 精度，但 CUTLASS 没有对应封装——自己写！

### 4.1 揭开 MMA Atom 的面纱

一个 MMA op 要回答两大问题：①选哪条 PTX 指令；②元素/寄存器映射关系。

**MMA op**：封装特定 PTX 指令（每个线程持有几个寄存器，直接写在类型里）：

In [ ]:
// SM80_16x8x8_F16F16F16F16_TN：Ampere 上的 mma.sync 封装
// 命名：SM80=Ampere, 16x8x8=M×N×K, F16F16F16F16=D/A/B/C 均 fp16, TN=A 行主序/B 列主序
struct SM80_16x8x8_F16F16F16F16_TN {
    using DRegisters = uint32_t[2];  // 16x8 / 32 线程 = 每线程 4 个 fp16 = 2 个寄存器
    using ARegisters = uint32_t[2];  // 16x8 / 32 = 4 个 fp16 = 2 个寄存器
    using BRegisters = uint32_t[1];  // 8x8 / 32 = 2 个 fp16 = 1 个寄存器
    using CRegisters = uint32_t[2];  // 与 D 相同

    CUTE_HOST_DEVICE static void
    fma(uint32_t& d0, uint32_t& d1,
        uint32_t const& a0, uint32_t const& a1,
        uint32_t const& b0,
        uint32_t const& c0, uint32_t const& c1) {
#if defined(CUTE_ARCH_MMA_SM80_ENABLED)
        asm volatile(
            "mma.sync.aligned.m16n8k8.row.col.f16.f16.f16.f16 "
            "{%0, %1},{%2, %3},{%4},{%5, %6};\n"
            : "=r"(d0), "=r"(d1)
            :  "r"(a0),  "r"(a1), "r"(b0), "r"(c0), "r"(c1));
#else
        CUTE_INVALID_CONTROL_PATH("Attempting to use SM80_16x8x8... without CUTE_ARCH_MMA_SM80_ENABLED");
#endif
    }
};

**MMA Traits**：描述指令内生的"线程/寄存器 → 矩阵坐标"映射（TV Layout）：

In [ ]:
template <>
struct MMA_Traits<SM80_16x8x8_F16F16F16F16_TN> {
    using ValTypeD = half_t;  using ValTypeA = half_t;
    using ValTypeB = half_t;  using ValTypeC = half_t;

    using Shape_MNK = Shape<_16, _8, _8>;
    using ThrID    = Layout<_32>;

    // ALayout: A 矩阵 (M=16,K=8) 元素在 32 线程寄存器中的分布
    // 线性索引 = 32*i + j + 16*p + 8*q
    using ALayout = Layout<Shape<Shape<_4,_8>, Shape<_2,_2>>,
                           Stride<Stride<_32,_1>, Stride<_16,_8>>>;
    using BLayout = Layout<Shape<Shape<_4,_8>, _2>,
                           Stride<Stride<_16,_1>, _8>>;
    using CLayout = Layout<Shape<Shape<_4,_8>, Shape<_2,_2>>,
                           Stride<Stride<_32,_1>, Stride<_16,_8>>>;
};

### 4.2 TV Layout 与 MN Layout

- **TV Layout**：`(线程ID, 元素index) -> (M, N)` 坐标，即 `(T, V) -> (M, N)`；
- **MN Layout**：TV Layout 的逆映射，`(M, N) -> (T, V)`（上篇展示的 MMA 映射图就是 MN Layout）。

![TV Layout 映射关系的含义](assets/figs/fig_05_TV_Layout_映射关系的含义.png)

推导示例：`ALayout = ((4,8),(2,2)) : ((32,1),(16,8))`，求线程 11 的第 2 个元素：

In [ ]:
(T, V) = (11, 2)
→ 把 11 按 shape (4,8) 拆成坐标: (11%4, 11/4) = (3, 2)
→ 嵌套坐标 ((3,2),(0,1))
→ 按 stride 计算 idx = 3×32 + 2×1 + 0×16 + 1×8 = 106
→ 按 shape (16,8) 拆成坐标: (106%16, 106/16) = (10, 6)

所以 `(T,V)=(11,2)` 映射到矩阵坐标 `(M,N)=(10,6)`，可在 MN Layout 图中确认。（严格说 A 的是 MK Layout、B 是 KN Layout，只有 C 才是正牌 MN Layout，本文统一称 MN Layout。）

### 4.3 写 FP8 MMA op：SM90_16x8x32_F32E4M3E5M2F32_TN

PTX 指令：`mma.sync.aligned.m16n8k32.row.col.f32.e4m3.e5m2.f32`

寄存器数计算：每线程 A=16 个（4 个 uint32 寄存器）、B=8 个（2 个）、C/D=4 个 FP32（4 个）：

In [ ]:
struct SM90_16x8x32_F32E4M3E5M2F32_TN {
    using DRegisters = float[4];
    using ARegisters = uint32_t[4];
    using BRegisters = uint32_t[2];
    using CRegisters = float[4];

    CUTE_HOST_DEVICE static void
    fma(float& d0, float& d1, float& d2, float& d3,
        uint32_t const& a0, uint32_t const& a1, uint32_t const& a2, uint32_t const& a3,
        uint32_t const& b0, uint32_t const& b1,
        float const& c0, float const& c1, float const& c2, float const& c3) {
#if defined(CUTE_ARCH_MMA_SM89_ENABLED)
        asm volatile(
            "mma.sync.aligned.m16n8k32.row.col.f32.e4m3.e5m2.f32 "
            "{%0, %1, %2, %3},{%4, %5, %6, %7},{%8, %9},{%10, %11, %12, %13};\n"
            : "=f"(d0), "=f"(d1), "=f"(d2), "=f"(d3)
            :  "r"(a0),  "r"(a1),  "r"(a2),  "r"(a3),
               "r"(b0),  "r"(b1),
               "f"(c0),  "f"(c1),  "f"(c2),  "f"(c3));
#else
        CUTE_INVALID_CONTROL_PATH("...without CUTE_ARCH_MMA_SM89_ENABLED");
#endif
    }
};

**小诀窍推导 TV Layout**：从 MN Layout 图数步长。以 T mode 为例：T0V0→T1V0 步长 64，T0V0→T4V0 步长 1，所以 T 有两个 sub-mode，Shape=(4,8)、Stride=(64,1)。同理推出：

In [ ]:
template <>
struct MMA_Traits<SM90_16x8x32_F32E4M3E5M2F32_TN> {
    using ValTypeD = float;  using ValTypeA = float_e4m3_t;
    using ValTypeB = float_e5m2_t;  using ValTypeC = float;

    using Shape_MNK = Shape<_16, _8, _32>;
    using ThrID    = Layout<_32>;
    using ALayout = Layout<Shape<Shape<_4,_8>, Shape<_4,_2,_2>>,
                           Stride<Stride<_64,_1>, Stride<_16,_8,_256>>>;
    using BLayout = Layout<Shape<Shape<_4,_8>, Shape<_4,_2>>,
                           Stride<Stride<_32,_1>, Stride<_8,_128>>>;
    using CLayout = Layout<Shape<Shape<_4,_8>, Shape<_2,_2>>,
                           Stride<Stride<_32,_1>, Stride<_16,_8>>>;
};

最后 `using MMA_op = SM90_16x8x32_F32E4M3E5M2F32_TN;` 即可。

![FP8 A 矩阵 MN Layout 示意图](assets/figs/fig_06_FP8_A_矩阵_MN_Layout_示意图.png)

### 4.4 验证与 PTX / SASS

8 个样例（4 类算子 × MM/MMA）全部 Success。PTX 就是我们内联汇编的指令；SASS 用一串 `F2FP.F16.E4M3.UNPACK_B` 解包 FP8，最终以 `HMMA.16816.F32` 计算。

## 同时回答

1. MMA 场景下涉及哪几种精度？为什么 `torch.matmul` / `torch.addmm` 无法满足混合精度需求？
2. 寄存器中转换精度用哪两个 CuTe API？`make_tensor_like` 做了什么？
3. 以 `ALayout = ((4,8),(2,2)) : ((32,1),(16,8))` 为例，说明 `(T,V)=(11,2)` 如何一步步映射到矩阵坐标？最终结果是多少？TV Layout 和 MN Layout 是什么关系？

把代码和三个答案发给我，我继续审查。